Made this notebook to check that I get the same results as Gallagher et al on their parameter estiamtes directly.
After that, we investigate if median stan draws line up with their estimates. 

In [29]:
import pandas as pd
import sys

sys.path.append("..")  

from Mathematical_Biomarkers.utils.myUtils import calc_average_overshoot, calc_delta_benefit, predict_ct_ttp, predict_drug_benefit


Have to redefine predict eTTP function since its not in myutils

In [ ]:
def predict_at_window_ttp(tau, n_crit, **kwargs):
    params = kwargs
    n_overshoot = calc_average_overshoot(tau, n_crit, **params)
    ave_n = ((n_crit + 1.2 * params['n0'])/2 - n_overshoot)
    benefit = calc_delta_benefit(kappa=(ave_n / params['K']), **params)
    ct_ttp = predict_ct_ttp(**params)
    return ct_ttp * (1 + benefit)

In [27]:
first_cycle_bruchovsky = pd.read_csv("../Mathematical_Biomarkers/paper_figures/patient_fits/first_cycle_params.csv")
first_cycle_bruchovsky

,index,PatientId,FitId,ModelName,AIC,BIC,RSquared,Success,Message,NumericalStabilisation,...,dD_ci,theta_ci,D_ci,DMax_ci,S0_ci,R0_ci,TimeInTrial,Cost,Turnover,Progression
0,0,12,0,LotkaVolterraModel,-49.132990,-47.716889,0.531877,True,`ftol` termination condition is satisfied.,False,...,0.0,0.0,0.0,0.0,0.0,0.0,1001,0.1,0.053555,True
1,0,20,0,LotkaVolterraModel,-52.743559,-51.077133,0.476434,True,`ftol` termination condition is satisfied.,False,...,0.0,0.0,0.0,0.0,0.0,0.0,1488,0.1,0.356593,True
2,0,25,0,LotkaVolterraModel,-54.256990,-51.986002,0.602269,True,`ftol` termination condition is satisfied.,False,...,0.0,0.0,0.0,0.0,0.0,0.0,1418,0.1,0.309963,True
3,0,78,0,LotkaVolterraModel,-71.010963,-68.739975,0.692827,True,`ftol` termination condition is satisfied.,False,...,0.0,0.0,0.0,0.0,0.0,0.0,1337,0.1,0.393746,True
4,0,85,0,LotkaVolterraModel,-76.846514,-74.575526,0.765899,True,`ftol` termination condition is satisfied.,False,...,0.0,0.0,0.0,0.0,0.0,0.0,728,0.1,0.228722,True


Biomarker values from the From table 8 in biomarker supplemental file

| Patient ID | AT Score | eMDD  | eTTP, d |
|:----------:|:--------:|:-----:|:-------:|
| 12         | 0.270    | 0.625 | 588     |
| 20         | 0.495    | 0.352 | 1144    |
| 25         | 0.439    | 0.362 | 1292    |
| 78         | 0.552    | 0.311 | 1235    |
| 85         | 0.366    | 0.449 | 907     |



In [42]:
ids = [12,20,25,78,85]
filtered_ids = first_cycle_bruchovsky[first_cycle_bruchovsky['PatientId'].isin(ids)] 

filtered_ids["at_score"] = calc_delta_benefit(kappa=.2, **filtered_ids)

exp_at = []
for pid in ids:
    row = filtered_ids[filtered_ids.PatientId == pid].iloc[0]
    exp_at.append(predict_at_window_ttp(tau=21, n_crit=0.5 * row.n0, **row.to_dict()))

exp_at

[np.float64(630.1115721405231),
 np.float64(1825.5427867307462),
 np.float64(1532.583156213254),
 np.float64(2159.4766690070387),
 np.float64(1199.8925744510611)]